# Module 3 pipeline walkthrough
Run the workflow in Anaconda Prompt first. Select the retail-m5 kernel. These cells help you inspect the actual prepared data rather than claim unperformed validation.

In [ ]:
from pathlib import Path
import sys, json, uuid
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.prepare import Audit
OUT = ROOT / 'data' / 'processed'
audit = Audit(OUT / 'privacy_audit.jsonl', 'learner-review-' + str(uuid.uuid4()))
audit.record('read', 'prepared.parquet', purpose='learner validation walkthrough')
df = pd.read_parquet(OUT / 'prepared.parquet')
df.shape

## Rows and unique keys
The expected grain is one product-store-date. There should be 78,600 rows and zero duplicate keys.

In [ ]:
print(df.groupby('split').size())
print('Duplicate keys:', df.duplicated(['id', 'date']).sum())
print('Negative labels:', df['units'].lt(0).sum())

## Missing values need interpretation
Compare training and held-out records separately. Optional event fields are nullable. Held-out lags and prices are deliberately unavailable after the fixed forecast cutoff.

In [ ]:
df.groupby('split')[['units', 'lag_7', 'lag_28', 'rolling_mean_28', 'historical_sell_price']].apply(lambda x: x.isna().sum())

## Representation
Count distinct series rather than daily rows. Discuss why CA_4 and HOUSEHOLD_2 require review before generalising model performance.

In [ ]:
print(df.groupby('store_id')['id'].nunique())
print(df.groupby('dept_id')['id'].nunique())
audit.record('read', 'bias_report.json', purpose='representation review')
bias = json.loads((OUT / 'bias_report.json').read_text())
bias['flags']

## Great Expectations and lineage
Results come from an executed suite. A passing quality suite does not prove a useful forecasting model or lower inventory cost.

In [ ]:
audit.record('read', 'gx_validation.json', purpose='quality suite review')
quality = json.loads((OUT / 'gx_validation.json').read_text())
audit.record('read', 'lineage.json', purpose='reproducibility review')
lineage = json.loads((OUT / 'lineage.json').read_text())
print(quality['statistics'])
print(lineage['source_sha256'])
print(lineage['versions'])